# EXP-20260828-vector-01

```text
실험 ID: EXP-20260828-vector-01
생성 기준일: 2026-08-28
기준 소스: src 최신본
실험 목적: PDF 적재와 Vector 검색 단독 검증 (로컬 pgvector)
```

이 노트북은 생성 당시 `src/` 의 복사본을 가진 독립 실험 공간이다 (계획 v4 §3).
`%%module` 셀 수정은 `src/` 에 자동 반영되지 않으며, `sync_to_py(dry_run=False)` 는 최종 채택 시에만 실행한다.

**계획과의 차이(사유)**: 계획 §6의 `tools.data_api.FinancialDataClient` 는 저장소에 구현이 없다.
Azure `/db` 는 2026-08-29 만료 + 원격 vec 테이블 0행이므로(사용자 결정: 로컬 테스트)
같은 호출 형태를 유지한 채 **로컬 pgvector** 로 구현해 `tools.data_api` 모듈로 등록한다.

In [1]:
# === 셀 매직 정의: 각 셀을 실제 모듈로 등록한다 ===
# 사용법: 셀 첫 줄에 `%%module <모듈명> <src 기준 경로>`.
# 셀을 수정하고 재실행하면 sys.modules 가 교체되므로,
# 그 모듈을 import 하는 하위 셀들을 다시 실행하면 수정본이 반영된다.
import json as _json
import sys as _sys
import types as _types
from pathlib import Path

from IPython.core.magic import register_cell_magic

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
NB_PATH = REPO_ROOT / "test" / "notebook" / "experiments" / "agent_vector_0828.ipynb"
RESULTS_DIR = REPO_ROOT / "test" / "notebook" / "experiments" / "results"
RESULTS_DIR.mkdir(exist_ok=True)


@register_cell_magic("module")
def _module_magic(line, cell):
    name, relpath = line.split()
    mod = _types.ModuleType(name)
    mod.__file__ = str(REPO_ROOT / "src" / relpath)
    _sys.modules[name] = mod
    parts = name.split(".")
    for i in range(1, len(parts)):
        pkg = ".".join(parts[:i])
        parent = _sys.modules.setdefault(pkg, _types.ModuleType(pkg))
        setattr(parent, parts[i], _sys.modules.get(name) if i == len(parts) - 1 else _sys.modules.setdefault(".".join(parts[:i + 1]), _types.ModuleType(".".join(parts[:i + 1]))))
    exec(compile(cell, mod.__file__, "exec"), mod.__dict__)
    print(f"registered: {name}")


def sync_to_py(dry_run=True):
    """%%module 셀을 src/*.py 로 되쓴다. 최종 채택 시에만 사용 (계획 v4 §10)."""
    nb = _json.loads(NB_PATH.read_text(encoding="utf-8"))
    for c in nb["cells"]:
        src = "".join(c["source"])
        if c["cell_type"] != "code" or not src.startswith("%%module "):
            continue
        first, _, body = src.partition("\n")
        _, name, relpath = first.split()
        target = REPO_ROOT / "src" / relpath
        old = target.read_text(encoding="utf-8") if target.exists() else None
        if old == body:
            print(f"  same: {relpath}")
        elif dry_run:
            print(f"CHANGED: {relpath}  (dry_run — 반영하려면 sync_to_py(dry_run=False))")
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_text(body, encoding="utf-8")
            print(f"WROTE: {relpath}")


## `src/config.py` / `src/clova.py` (verbatim)

In [2]:
%%module config config.py
# -*- coding: utf-8 -*-
"""경로·모델 상수. 채권 MVP 범위."""
from pathlib import Path

# config.py 는 src/ 안에 있다. .parent = src/, 한 번 더 올려야 저장소 루트다.
# 한 번만 올리면 ARTIFACTS 가 src/artifacts 를 새로 만들며 조용히 캐시를 잃는다.
ROOT = Path(__file__).resolve().parent.parent

# 지시서는 ontology/bond.ttl 하나를 가정하지만 이 저장소의 스키마는 common + 도메인 4로 갈려 있다.
# 테스트 질문의 '위험등급'(fp:RiskGrade·fp:riskGradeLevel)은 common.ttl에만 있어서
# bond_kr.ttl만 인덱싱하면 4문항 중 1문항을 못 답한다. 둘 다 넣는다.
BOND_TTL_PATHS = [ROOT / "ontology" / "bond_kr.ttl", ROOT / "ontology" / "common.ttl"]

ARTIFACTS = ROOT / "artifacts"

# 스키마 벡터 인덱스는 PostgreSQL + pgvector 에 둔다(FAISS 에서 이전, 2026-08-22).
# 이전 근거는 vectordb_test/results/1_pgvector_test_report.md — cosine 점수가
# FAISS(정규화 후 IndexFlatIP)와 최대 오차 5.03e-07 로 일치해 임계값을 그대로 쓴다.
# 비밀번호를 포함하므로 DSN 을 로그에 찍지 않는다. 접속 정보는 환경변수로 덮을 수 있다.
import os

# Azure Data API
# 현재 공개 테스트 API는 임시 주소다. 운영에서는 환경변수로 반드시 덮어쓴다.
FINANCIAL_DATA_API_URL = os.environ.get(
    "FINANCIAL_DATA_API_URL",
    "http://40.82.145.44:8000",
)
FINANCIAL_DATA_RELEASE_ID = os.environ.get(
    "FINANCIAL_DATA_RELEASE_ID",
    "financial-products-2026-08-24@"
    "ddb3d994a4a5115a75bed7efa9c4cd0f6655f95b0a49f3b0e3c01b2bf8301a38",
)
DATA_API_TIMEOUT_SECONDS = float(
    os.environ.get("DATA_API_TIMEOUT_SECONDS", "10")
)

# Direct PostgreSQL connection.
# Existing rdb/bond_schema tools still use this configuration.
BOND_DB = {
    "host": os.environ.get("PGHOST", "127.0.0.1"),
    "port": os.environ.get("PGPORT", "5432"),
    "user": os.environ.get("PGUSER", "postgres"),
    "password": os.environ.get("PGPASSWORD", "postgres"),
    "dbname": os.environ.get("PGDATABASE", "mafest"),
}
# 유닉스 소켓은 peer 인증에 걸린다. host 를 명시해 TCP 로 붙는다.
BOND_DSN = " ".join(f"{k}={v}" for k, v in BOND_DB.items())
BOND_TABLE = "bond_schema_terms"
EMBED_DIM = 1024

BOND_TOP_K = 5
# cosine 점수 하한. 실측상 0.36~0.38대는 무관한 용어(자회사 관계 등)가 섞인다.
# 빈약한 근거를 주면 모델이 일반 지식으로 메워 근거 없는 단정이 나온다.
BOND_SCORE_FLOOR = 0.45

EMBEDDING_MODEL = "bge-m3"        # 1024차원, cosine (CLOVA Studio)
# 1단계 Query Frame 추출. HCX-005·HCX-DASH-002 와 대표 4문항으로 비교해 정했다 —
# 스키마 준수 4/4 vs 2/4 vs 1/4. DASH-002 의 속도 이점은 프롬프트가 길어지면서
# 사라졌다(출력 토큰이 지연을 지배한다). 근거: vectordb_test/4_query_frame_v1/4_result_query_frame_v1.md
FRAME_MODEL = "HCX-007"
ANSWER_MODEL = "HCX-005"          # 답변 생성
CHAT_TIMEOUT_SECONDS = 13           # API tail stall은 재시도 없이 ABSTAIN해 15초 E2E를 지킨다

CLOVA_HOST = "https://clovastudio.stream.ntruss.com"


registered: config


In [3]:
%%module clova clova.py
# -*- coding: utf-8 -*-
"""CLOVA Studio 클라이언트 — 채팅과 임베딩.

콘솔 샘플 코드와 다른 점 셋 (실측으로 확인):
  1. Accept를 application/json 으로. text/event-stream 이면 SSE라 반환값으로 못 쓴다.
  2. 추론 모델(HCX-007)은 maxTokens 를 거부한다. maxCompletionTokens 를 쓴다
     — 값 범위 문제가 아니라 파라미터명이 다르다(maxTokens=4096도 40001).
  3. HCX-007에서 structured outputs 를 쓰려면 thinking 을 명시적으로 꺼야 한다.
     기본 ON 이라 responseFormat 과 충돌한다. "off"는 무효값이고 "none"만 받는다.
"""
import json
import sys

import requests

from config import CHAT_TIMEOUT_SECONDS, CLOVA_HOST, EMBEDDING_MODEL, ROOT

REASONING_MODELS = {"HCX-007"}


def load_key() -> str:
    """.env의 clova 키. 값은 절대 로그에 남기지 않는다."""
    env = ROOT / ".env"
    if not env.exists():
        sys.exit(f"FAIL  .env 없음: {env}")
    for line in env.read_text(encoding="utf-8").splitlines():
        k, _, v = line.partition("=")
        if k.strip() == "clova":
            key = v.strip().strip('"').strip("'")
            if not key:
                sys.exit("FAIL  .env의 clova 값이 비어 있음")
            return key if key.startswith("Bearer ") else f"Bearer {key}"
    sys.exit("FAIL  .env에 clova 항목 없음")


_KEY = None


def key() -> str:
    global _KEY
    if _KEY is None:
        _KEY = load_key()
    return _KEY


def chat(model, system, user, max_tokens=1024, response_format=None, temperature=0.1):
    body = {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": system}]},
            {"role": "user", "content": [{"type": "text", "text": user}]},
        ],
        "topP": 0.8, "temperature": temperature, "repetitionPenalty": 1.1,
        "stop": [], "seed": 0,
    }
    if model in REASONING_MODELS:
        body["maxCompletionTokens"] = max_tokens
        if response_format:
            body["thinking"] = {"effort": "none"}
    else:
        body["maxTokens"] = max_tokens
        body["topK"] = 0
        body["includeAiFilters"] = True
    if response_format:
        body["responseFormat"] = response_format

    r = requests.post(f"{CLOVA_HOST}/v3/chat-completions/{model}",
                      headers={"Authorization": key(),
                               "Content-Type": "application/json; charset=utf-8",
                               "Accept": "application/json"},
                      json=body, timeout=CHAT_TIMEOUT_SECONDS)
    if r.status_code != 200:
        # 본문을 삼키면 원인을 못 찾는다. 40001 메시지에 어느 파라미터인지 들어 있다.
        raise RuntimeError(f"HTTP {r.status_code} — {r.text[:220]}")
    data = r.json()
    code = (data.get("status") or {}).get("code")
    if code not in (None, "20000"):
        raise RuntimeError(f"status {code} — {(data.get('status') or {}).get('message')}")
    content = (data.get("result") or {}).get("message", {}).get("content")
    if isinstance(content, list):   # v3는 입력이 배열이라 출력도 배열로 오는 경우가 있다
        content = "".join(p.get("text", "") for p in content if isinstance(p, dict))
    if not isinstance(content, str):
        raise RuntimeError(f"content 형태 불명: {type(content)}")
    return content


def parse_json_loose(text: str) -> dict:
    """모델이 코드펜스나 설명을 붙여도 JSON 객체만 뽑는다."""
    s = text.strip()
    if s.startswith("```"):
        s = s.split("```")[1] if "```" in s[3:] else s[3:]
        s = s.removeprefix("json").strip()
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1:
        raise json.JSONDecodeError("객체를 못 찾음", s, 0)
    return json.loads(s[a:b + 1])


_EMB = None


def _embedder():
    global _EMB
    if _EMB is None:
        from langchain_naver import ClovaXEmbeddings
        _EMB = ClovaXEmbeddings(model=EMBEDDING_MODEL,
                                api_key=key().removeprefix("Bearer ").strip())
    return _EMB


def _embed_raw(text: str) -> list[float]:
    """캐시를 거치지 않는 실제 API 호출. 이 함수만 embed_query를 부른다.

    embed_many가 이걸 부르고 embed는 embed_many에 위임한다. 셋 중 하나라도
    서로를 부르면 무한 재귀가 된다 — 실제로 embed_many가 embed를 부르던 시절
    캐시 미스에서 RecursionError가 났다.
    """
    return _embedder().embed_query(text)


def embed(text: str) -> list[float]:
    """단건 임베딩. embed_many에 위임해 디스크 캐시를 공유한다.

    직접 embed_query를 부르면 캐시를 지나치므로, 같은 질문이 반복될 때마다
    API를 다시 때리고 간격도 없어 연속 호출 시 429에 걸린다.
    임베딩은 같은 텍스트·모델이면 결정적이라 캐시해도 값이 달라지지 않는다.
    """
    return embed_many([text], pause=0.0, progress=False)[0]


def embed_many(texts: list[str], pause: float = 1.2, progress: bool = True) -> list[list[float]]:
    """여러 건 임베딩. 분당 쿼터가 있어 간격을 두고, 결과는 디스크에 캐시한다.

    - langchain의 embed_documents는 지연 없이 연속 호출해 429(rate exceeded)를 맞는다.
      실측상 약 60건 연속이면 차단되므로 기본 간격을 1.2s(≈50건/분)로 둔다.
    - 캐시가 없으면 중간에 실패할 때 앞서 성공한 호출이 통째로 버려진다.
      TTL 주석은 앞으로 계속 손볼 예정이라 재빌드가 반복된다.
    """
    import hashlib
    import time as _t

    from config import ARTIFACTS
    ARTIFACTS.mkdir(exist_ok=True)
    cache_path = ARTIFACTS / "embed_cache.json"
    cache = {}
    if cache_path.exists():
        try:
            cache = json.loads(cache_path.read_text(encoding="utf-8"))
        except Exception:
            cache = {}      # 깨진 캐시는 조용히 버린다. 다시 만들면 그만이다.

    def kk(t):
        return hashlib.sha1(f"{EMBEDDING_MODEL}\x00{t}".encode()).hexdigest()

    out, new_hits, api_calls = [], 0, 0
    for i, t in enumerate(texts):
        h = kk(t)
        if h in cache:
            out.append(cache[h])
            continue
        for attempt in range(7):
            try:
                v = _embed_raw(t)      # embed() 를 부르면 여기로 되돌아와 무한 재귀가 된다
                break
            except Exception as e:
                if "429" not in str(e) and "42901" not in str(e):
                    raise
                wait = min(2 ** attempt, 65)      # 분 단위 쿼터라 최대 65s까지 기다린다
                if progress:
                    print(f"  429 — {wait}s 대기 ({i+1}/{len(texts)})", flush=True)
                _t.sleep(wait)
        else:
            cache_path.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
            raise RuntimeError(f"429 재시도 7회 실패: {i}번째 (여기까지는 캐시에 저장됨)")
        out.append(v)
        cache[h] = v
        new_hits += 1
        api_calls += 1
        if new_hits % 20 == 0:      # 중간 저장 — 크래시해도 진행분이 남는다
            cache_path.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
        if progress and (i + 1) % 25 == 0:
            print(f"  {i+1}/{len(texts)}  (API 호출 {api_calls})", flush=True)
        _t.sleep(pause)

    cache_path.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
    if progress:
        print(f"  임베딩 완료 — API 호출 {api_calls}건 / 캐시 재사용 {len(texts)-api_calls}건")
    return out


if __name__ == "__main__":
    # 자기검사: embed / embed_many / _embed_raw 의 호출 고리가 닫히지 않았는지 본다.
    # 캐시 적중 경로에서는 재귀가 드러나지 않으므로 반드시 '캐시에 없는' 문장을 쓴다.
    # (실제로 embed_many 가 embed 를 부르던 시절 RecursionError 가 났고,
    #  회귀 테스트 72건이 전부 캐시 적중이라 그 버그를 못 잡았다.)
    import uuid
    probe = f"clova 자기검사 {uuid.uuid4()}"
    v = embed(probe)
    assert len(v) == 1024, f"차원 이상: {len(v)}"
    assert embed(probe) == v, "같은 문장인데 결과가 다르다 — 캐시가 안 먹는다"
    print(f"clova 자기검사 PASS — 캐시미스 경로 dim={len(v)}, 재호출 일치")


registered: clova


## `tools.data_api` (신규 — 노트북에서만 존재, 채택 시 `src/tools/data_api.py` 로 승격)

In [4]:
%%module tools.data_api tools/data_api.py
# -*- coding: utf-8 -*-
"""로컬 pgvector 콘텐츠 인덱스(vec.document_chunk) 적재·검색.

계획서의 `tools.data_api.FinancialDataClient` 는 Azure Data API 클라이언트로 문서에만 있고
구현이 없다. Azure `/db` 는 2026-08-29 만료 + 원격 vec 테이블 0행이므로(사용자 결정)
동일한 호출 형태(`FinancialDataClient.from_env().semantic_search(vec, top_k)`)를
로컬 pgvector 로 구현한다. DDL 은 Azure vec.document_chunk 12컬럼을 미러링한다.
"""
import hashlib
from pathlib import Path

import psycopg

import clova
from config import BOND_DSN

EMBED_DIM = 1024
SCORE_FLOOR = 0.45
DATA_CUTOFF = "2026-08-24"

DDL = """
CREATE SCHEMA IF NOT EXISTS vec;
CREATE EXTENSION IF NOT EXISTS vector;
CREATE TABLE IF NOT EXISTS vec.document_chunk (
  chunk_id        text PRIMARY KEY,
  document_id     text NOT NULL,
  product_id      text,
  page_number     integer,
  citation_text   text NOT NULL,
  chunk_text      text NOT NULL,
  published_at    date NOT NULL CHECK (published_at <= DATE '2026-08-24'),
  source_url      text NOT NULL,
  content_hash    text NOT NULL,
  embedding_model text NOT NULL,
  embedding_dim   smallint NOT NULL,
  embedding       vector(1024) NOT NULL
);
CREATE INDEX IF NOT EXISTS document_chunk_embedding_hnsw
  ON vec.document_chunk USING hnsw (embedding vector_cosine_ops);
"""

PRODUCT_TABLES = {"fund_pub": ("raw.fund_pub_master", "itm_no"),
                  "etf_kr": ("raw.etf_kr_master", "pd_itm_no")}


def _conn():
    return psycopg.connect(BOND_DSN, autocommit=True)


def _pages(pdf_path: Path) -> list:
    from pypdf import PdfReader
    reader = PdfReader(str(pdf_path))
    return [(page.extract_text() or "").strip() for page in reader.pages]


def _chunks(text: str, limit: int = 2000, piece: int = 1200) -> list:
    # ponytail: 1페이지=1청크, 2000자 초과만 문단 경계 분할. 검색 품질 부족이 실측되면 슬라이딩 윈도우.
    if len(text) <= limit:
        return [text] if text else []
    out, buf = [], ""
    for para in text.split("\n"):
        if len(buf) + len(para) + 1 > piece and buf:
            out.append(buf.strip())
            buf = ""
        buf += para + "\n"
    if buf.strip():
        out.append(buf.strip())
    return out


def _assert_product(cur, meta):
    """상품코드가 로컬 RDB 에 실재하는지 검증 — 조용한 오매핑 방지 (빌드 게이트)."""
    table, col = PRODUCT_TABLES[meta["domain"]]
    cur.execute(f"SELECT count(*) FROM {table} WHERE {col} = %s", (meta["product_id"],))
    n = cur.fetchone()[0]
    assert n > 0, f"RDB 에 없는 상품코드: {meta['product_id']} ({table}.{col})"


def ingest(pdf_dir, manifest: dict) -> dict:
    """manifest: {파일명: {product_id, domain, published_at}}. TRUNCATE 후 재적재(멱등)."""
    rows = []
    with _conn() as conn, conn.cursor() as cur:
        cur.execute(DDL)
        for fname, meta in manifest.items():
            path = Path(pdf_dir) / fname
            assert path.is_file(), f"PDF 없음: {path}"
            assert str(meta["published_at"]) <= DATA_CUTOFF, f"look-ahead: {fname}"
            _assert_product(cur, meta)
            doc_id = path.stem
            for pno, page_text in enumerate(_pages(path), start=1):
                for i, chunk in enumerate(_chunks(page_text)):
                    rows.append({
                        "chunk_id": f"{doc_id}:p{pno}:{i}",
                        "document_id": doc_id,
                        "product_id": meta["product_id"],
                        "page_number": pno,
                        "citation_text": f"{doc_id} p.{pno}",
                        "chunk_text": chunk,
                        "published_at": meta["published_at"],
                        "source_url": path.resolve().as_uri(),
                        "content_hash": hashlib.sha256(chunk.encode()).hexdigest(),
                    })
        assert rows, "추출된 텍스트 청크가 없다 — PDF 텍스트 추출 실패 여부를 확인할 것"
        vectors = clova.embed_many([r["chunk_text"] for r in rows])
        cur.execute("TRUNCATE vec.document_chunk")
        cur.executemany(
            "INSERT INTO vec.document_chunk (chunk_id, document_id, product_id, page_number,"
            " citation_text, chunk_text, published_at, source_url, content_hash,"
            " embedding_model, embedding_dim, embedding)"
            " VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,'bge-m3',1024,%s::vector)",
            [(r["chunk_id"], r["document_id"], r["product_id"], r["page_number"],
              r["citation_text"], r["chunk_text"], r["published_at"], r["source_url"],
              r["content_hash"], str([float(x) for x in v]))
             for r, v in zip(rows, vectors)])
        cur.execute("SELECT count(*), count(DISTINCT document_id),"
                    " max(vector_dims(embedding)) FROM vec.document_chunk")
        n, docs, dim = cur.fetchone()
    assert n == len(rows) and dim == EMBED_DIM, (n, dim)
    return {"chunks": n, "documents": docs, "dim": dim}


def index_count() -> int:
    try:
        with _conn() as conn, conn.cursor() as cur:
            cur.execute("SELECT count(*) FROM vec.document_chunk")
            return cur.fetchone()[0]
    except psycopg.errors.UndefinedTable:
        return 0


class FinancialDataClient:
    """계획서 호출 형태 유지 어댑터 — 로컬 pgvector 조회."""

    @classmethod
    def from_env(cls):
        return cls()

    def semantic_search(self, query_vector, top_k: int = 3) -> dict:
        vec_literal = str([float(x) for x in query_vector])
        try:
            with _conn() as conn, conn.cursor() as cur:
                cur.execute("SELECT count(*) FROM vec.document_chunk")
                if cur.fetchone()[0] == 0:
                    return {"status": "pending", "results": [], "raw_top": [],
                            "reason": "content index 미구축 — 근거 없음"}
                cur.execute(
                    "SELECT document_id, product_id, page_number, citation_text, chunk_text,"
                    " published_at, 1 - (embedding <=> %(v)s::vector) AS score"
                    " FROM vec.document_chunk"
                    " ORDER BY embedding <=> %(v)s::vector LIMIT %(k)s",
                    {"v": vec_literal, "k": top_k})
                raw = [{"document_id": r[0], "product_id": r[1], "page_number": r[2],
                        "title": r[3], "quote": r[4][:300],
                        "published_at": str(r[5]), "effective_as_of": str(r[5]),
                        "score": round(float(r[6]), 4)}
                       for r in cur.fetchall()]
        except psycopg.errors.UndefinedTable:
            return {"status": "pending", "results": [], "raw_top": [],
                    "reason": "vec.document_chunk 없음"}
        except psycopg.Error as exc:
            return {"status": "error", "results": [], "raw_top": [], "reason": str(exc)}
        hits = [h for h in raw if h["score"] >= SCORE_FLOOR]
        return {"status": "ok" if hits else "empty", "results": hits,
                "raw_top": [{"document_id": h["document_id"], "score": h["score"]} for h in raw]}


registered: tools.data_api


# PDF 적재 (계획 v4 §6)

PDF 읽기 → 텍스트 추출(pypdf) → 임베딩(bge-m3) → VectorDB 적재

In [5]:
# data/pdf_demo 의 실제 PDF 2개 ↔ 로컬 RDB 실재 상품코드 매핑 (사용자 지정)
# - 국민성장펀드: raw.fund_pub_master.itm_no, 클래스 4개(KR5153480100~103) 중 대표 클래스 종류C
# - TIGER MSCI Korea TR: raw.etf_kr_master.pd_itm_no
PDF_DIR = REPO_ROOT / "data" / "pdf_demo"
MANIFEST = {
    "국민참여형 국민성장펀드.pdf": {
        "product_id": "KR5153480100", "domain": "fund_pub", "published_at": "2026-08-24"},
    "미래에셋TIGERMSCIKOREATotalReturn증권상장지수투자신탁(주식).pdf": {
        "product_id": "KR7310970009", "domain": "etf_kr", "published_at": "2026-08-24"},
}

In [6]:
from tools import data_api

# 멱등 적재: 이미 적재돼 있으면 건너뛴다 (임베딩은 artifacts/embed_cache.json 캐시로 재실행도 저렴)
n = data_api.index_count()
if n == 0:
    print("ingest:", data_api.ingest(PDF_DIR, MANIFEST))
else:
    print(f"index 이미 적재됨: {n} chunks — 재적재하려면 data_api.ingest(PDF_DIR, MANIFEST)")

  임베딩 완료 — API 호출 9건 / 캐시 재사용 0건
ingest: {'chunks': 9, 'documents': 2, 'dim': 1024}


# Vector 테스트 질문 V01~V08

In [7]:
VECTOR_TEST_QUESTIONS = [
    {
        "id": "V01",
        "question": "국민참여형 국민성장펀드는 어떤 모펀드와 자펀드 구조로 운용되나요?",
        "expected_document": "국민참여형 국민성장펀드",
    },
    {
        "id": "V02",
        "question": "국민참여형 국민성장펀드에서 재정은 손실을 어떻게 우선 부담하나요?",
        "expected_document": "국민참여형 국민성장펀드",
    },
    {
        "id": "V03",
        "question": "국민참여형 국민성장펀드의 주요 투자 대상 산업과 투자 비율을 알려줘",
        "expected_document": "국민참여형 국민성장펀드",
    },
    {
        "id": "V04",
        "question": "국민참여형 국민성장펀드의 자펀드 운용사는 몇 곳이 선정되었나요?",
        "expected_document": "국민참여형 국민성장펀드",
    },
    {
        "id": "V05",
        "question": "TIGER MSCI Korea TR ETF가 추종하는 지수와 기초자산은 무엇인가요?",
        "expected_document": "TIGER MSCI Korea TR",
    },
    {
        "id": "V06",
        "question": "TIGER MSCI Korea TR의 상위 편입 종목과 종목별 비중을 알려줘",
        "expected_document": "TIGER MSCI Korea TR",
    },
    {
        "id": "V07",
        "question": "TIGER MSCI Korea TR 투자 시 발생할 수 있는 주요 위험과 주의사항은 무엇인가요?",
        "expected_document": "TIGER MSCI Korea TR",
    },
    {
        "id": "V08",
        "question": "Kimi 관련 투자상품 정보를 공식 문서에서 찾아줘",
        "expected_document": None,
    },
]

In [8]:
import clova
from tools.data_api import FinancialDataClient

client = FinancialDataClient.from_env()

VECTOR_RESULTS = []

for item in VECTOR_TEST_QUESTIONS:
    query_vector = clova.embed(item["question"])
    result = client.semantic_search(query_vector, top_k=3)

    VECTOR_RESULTS.append({
        "id": item["id"],
        "question": item["question"],
        "expected_document": item["expected_document"],
        "result": result,
    })

    print(f"[{item['id']}] {item['question']}")
    print(" status:", result["status"], "| top:",
          [(h["document_id"][:20], h["score"]) for h in result.get("raw_top", [])])
    print("-" * 80)

[V01] 국민참여형 국민성장펀드는 어떤 모펀드와 자펀드 구조로 운용되나요?
 status: ok | top: [('국민참여형 국민성장펀드', 0.6917), ('국민참여형 국민성장펀드', 0.6724), ('국민참여형 국민성장펀드', 0.6045)]
--------------------------------------------------------------------------------


[V02] 국민참여형 국민성장펀드에서 재정은 손실을 어떻게 우선 부담하나요?
 status: ok | top: [('국민참여형 국민성장펀드', 0.6124), ('국민참여형 국민성장펀드', 0.544), ('국민참여형 국민성장펀드', 0.5321)]
--------------------------------------------------------------------------------


[V03] 국민참여형 국민성장펀드의 주요 투자 대상 산업과 투자 비율을 알려줘
 status: ok | top: [('국민참여형 국민성장펀드', 0.6814), ('국민참여형 국민성장펀드', 0.6576), ('국민참여형 국민성장펀드', 0.574)]
--------------------------------------------------------------------------------


[V04] 국민참여형 국민성장펀드의 자펀드 운용사는 몇 곳이 선정되었나요?
 status: ok | top: [('국민참여형 국민성장펀드', 0.7199), ('국민참여형 국민성장펀드', 0.6357), ('국민참여형 국민성장펀드', 0.5948)]
--------------------------------------------------------------------------------


[V05] TIGER MSCI Korea TR ETF가 추종하는 지수와 기초자산은 무엇인가요?
 status: ok | top: [('미래에셋TIGERMSCIKOREATo', 0.6868), ('국민참여형 국민성장펀드', 0.4039), ('국민참여형 국민성장펀드', 0.3434)]
--------------------------------------------------------------------------------


[V06] TIGER MSCI Korea TR의 상위 편입 종목과 종목별 비중을 알려줘
 status: ok | top: [('미래에셋TIGERMSCIKOREATo', 0.7039), ('국민참여형 국민성장펀드', 0.4151), ('국민참여형 국민성장펀드', 0.3685)]
--------------------------------------------------------------------------------


[V07] TIGER MSCI Korea TR 투자 시 발생할 수 있는 주요 위험과 주의사항은 무엇인가요?
 status: ok | top: [('미래에셋TIGERMSCIKOREATo', 0.6379), ('국민참여형 국민성장펀드', 0.4171), ('국민참여형 국민성장펀드', 0.3967)]
--------------------------------------------------------------------------------


[V08] Kimi 관련 투자상품 정보를 공식 문서에서 찾아줘
 status: empty | top: [('국민참여형 국민성장펀드', 0.4299), ('국민참여형 국민성장펀드', 0.4251), ('미래에셋TIGERMSCIKOREATo', 0.4234)]
--------------------------------------------------------------------------------


# 성공 기준 평가·기록

In [9]:
import json

# 성공 기준 (계획 v4 §6):
#   V01~V04 → 국민성장펀드 문서가 Top-3 에 포함
#   V05~V07 → TIGER MSCI Korea TR 문서가 Top-3 에 포함
#   V08     → 검색 결과가 없거나 score threshold 미달
DOC_KEY = {"국민참여형 국민성장펀드": "국민성장",
           "TIGER MSCI Korea TR": "tigermscikorea"}


def matches(expected, doc_id):
    return DOC_KEY[expected] in doc_id.replace(" ", "").casefold()


records = []
for r in VECTOR_RESULTS:
    res = r["result"]
    top3 = [h["document_id"] for h in res.get("results", [])]
    if r["expected_document"] is None:
        verdict = "PASS" if res["status"] in ("empty", "pending") or not top3 else "FAIL"
    else:
        verdict = "PASS" if any(matches(r["expected_document"], d) for d in top3) else "FAIL"
    records.append({
        "experiment_id": "EXP-20260828-vector-01",
        "question_id": r["id"],
        "question": r["question"],
        "status": res["status"],
        "verdict": verdict,
        "retrieved_documents": res.get("results", []),
        "raw_top": res.get("raw_top", []),
    })
    print(f"[{r['id']}] {verdict}  status={res['status']}")

out_path = RESULTS_DIR / "vector_0828.json"
out_path.write_text(json.dumps(records, ensure_ascii=False, indent=2, default=str),
                    encoding="utf-8")
print("saved:", out_path)

[V01] PASS  status=ok
[V02] PASS  status=ok
[V03] PASS  status=ok
[V04] PASS  status=ok
[V05] PASS  status=ok
[V06] PASS  status=ok
[V07] PASS  status=ok
[V08] PASS  status=empty
saved: /mnt/c/Users/rladl/Desktop/2026_MIRAE_ASSET_AI-Festival/2026_10th_MIRAE-ASSET_AI-Festival/test/notebook/experiments/results/vector_0828.json
